<a href="https://colab.research.google.com/github/tomonari-masada/course2026-sml/blob/main/08_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ロジスティック回帰による糖尿病の予測

* 有名なPima Indians Diabetes Databaseを使う（下リンク先）

  * https://www.kaggle.com/uciml/pima-indians-diabetes-database

* ロジスティック回帰、そして、分類の評価については、下記も参照
  * https://developers.google.com/machine-learning/crash-course/logistic-regression/
  * https://developers.google.com/machine-learning/crash-course/classification/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    roc_auc_score, RocCurveDisplay,
    average_precision_score, PrecisionRecallDisplay
)
from sklearn.model_selection import StratifiedKFold

%config InlineBackend.figure_format = 'retina'

## データの読み込み

In [ ]:
diabetes = pd.read_csv('/content/drive/MyDrive/data/diabetes.csv')

In [ ]:
diabetes.head()

In [ ]:
y = diabetes['Outcome']
X = diabetes.drop('Outcome', axis=1)

## 訓練データ、テストデータに分割

**この分割は変えないようにしてください。**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=123)

In [ ]:
X_train.describe()

In [ ]:
X_train.hist(bins=50, figsize=(12,12));

## ベースライン: デフォルト設定のロジスティック回帰
* 交差検証も何もせずに、単にテストセット以外の部分で、モデルの学習を実行する。

In [ ]:
baseline = LogisticRegression(random_state=123)
baseline.fit(X_train, y_train)

* `max_iter`が小さいとの警告が出ているので、増やして学習しなおし。

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=123)
baseline.fit(X_train, y_train)

* 大丈夫だったので、テストデータでの最終評価値を得る。
 * scoreメソッドを使う。

In [ ]:
print(f'test score: {baseline.score(X_test, y_test):.4f}')

* Area under ROC curveも計算してみる。


In [ ]:
y_test_pred_proba = baseline.predict_proba(X_test)
print(f'ROC AUC: {roc_auc_score(y_test, y_test_pred_proba[:,1]):.4f}')

* ROC curveを描いてみる。
 * https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc.html#sphx-glr-auto-examples-model-selection-plot-roc-py

In [ ]:
display = RocCurveDisplay.from_estimator(baseline, X_test, y_test, name="baseline")
_ = display.ax_.set_title("ROC curve")

* precision-recall curveを描いてみる。

In [ ]:
display = PrecisionRecallDisplay.from_estimator(baseline, X_test, y_test, name="baseline")
_ = display.ax_.set_title("precision-recall curve")

In [ ]:
average_precision = average_precision_score(y_test, y_test_pred_proba[:,1])
print(f'average precision: {average_precision:.4f}')

* これをベースラインとみなす。
* これより良い結果を得るべく、試行錯誤する。
* 試行錯誤した結果として辿り着いたモデルで、**最後に一回、テストデータ上での評価**を行う。

---

**以下、訓練データ部分を使って、交差検証によって良いモデルを探す。**

* ここに示すのは一つの試行錯誤の例なので、これに従わなくても全然大丈夫です。
---



## 交差検証しつつ試行錯誤

* 元々の訓練データのコピーを作っておく。

In [ ]:
X_train_original = X_train.copy()

### 交差検証の準備

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)

* 交差検証のためのヘルパ関数

In [ ]:
def cv(skf, X_train, y_train, preprocess=None, max_iter=1000, **kwargs):

  # キーワード引数として、モデルの設定を指定できるようにしてある。
  for kwarg in kwargs:
    print(f'{kwarg} = {kwargs[kwarg]}')

  # 交差検証のループ
  scores = []
  for train_index, valid_index in skf.split(X_train, y_train):

    cv_X_train = X_train.iloc[train_index]
    cv_y_train = y_train.iloc[train_index]
    cv_X_valid = X_train.iloc[valid_index]
    cv_y_valid = y_train.iloc[valid_index]

    # データの前処理
    #   その都度、関数preprocessを定義してから、この関数cvを呼び出す。
    if preprocess:
      cv_X_train, cv_X_valid = preprocess(cv_X_train, cv_X_valid)

    # ロジスティック回帰の学習
    model = LogisticRegression(**kwargs, max_iter=max_iter)
    model.fit(cv_X_train, cv_y_train)

    # 検証データでの評価
    #score = model.score(cv_X_valid, cv_y_valid)
    score = average_precision_score(cv_y_valid, model.predict_proba(cv_X_valid)[:,1])
    print(f'average precision: {score:.4f}')
    scores.append(score)

  mean_score = np.array(scores).mean()
  print(f'mean average precision: {mean_score:.4f}')
  return mean_score

### デフォルトの設定での評価
* 交差検証で性能評価するとどうなるかを確認している。

In [ ]:
cv(skf, X_train, y_train);

### BloodPressureへの対応

* まず、属性「BloodPressure」について、ヒストグラムを描いてよくよく眺める。


In [ ]:
sns.histplot(X_train['BloodPressure']);

* 0という値がけっこうあるらしい。実は、これは欠測値。
* そこで、中央値で置き換えることにする。

In [ ]:
X_train_copy = X_train.copy()

feature = 'BloodPressure'
imp = SimpleImputer(missing_values=0, strategy='median')
X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

sns.histplot(X_train_copy[feature]);

* 欠測箇所を中央値で埋める関数を定義しておく。
 * これは、交差検証を実行するときに使用する。

In [ ]:
def preprocess(X_train, X_valid):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  feature = 'BloodPressure'
  X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
  X_valid_copy[feature] = imp.transform(X_valid[[feature]])
  print(f'  imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

* 交差検証で評価する。

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess);

### BMIへの対応

* 次に、training dataの「BMI」のヒストグラムを描いてみる


In [ ]:
sns.histplot(X_train['BMI']);

* やはり欠測部分が0とされているようなので、先ほどと同様、中央値で埋める。


In [ ]:
X_train_copy = X_train.copy()

feature = 'BMI'
imp = SimpleImputer(missing_values=0, strategy='median')
X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

sns.histplot(X_train_copy[feature]);

* 交差検証で評価する。
 * 欠測箇所を埋める関数を書き換える。

In [ ]:
def preprocess(X_train, X_valid):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  for feature in ['BloodPressure', 'BMI']:
    X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
    X_valid_copy[feature] = imp.transform(X_valid[[feature]])
    print(f'  imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess);

### Glucoseへの対応

In [ ]:
sns.histplot(X_train['Glucose']);

In [ ]:
X_train_copy = X_train.copy()

feature = 'Glucose'
imp = SimpleImputer(missing_values=0, strategy='median')
X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

sns.histplot(X_train_copy[feature]);

* 欠測箇所を埋める関数を書き換える。

In [ ]:
def preprocess(X_train, X_valid):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  for feature in ['BloodPressure', 'BMI', 'Glucose']:
    X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
    X_valid_copy[feature] = imp.transform(X_valid[[feature]])
    print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess);

* ここまでベストな結果は？

### SkinThicknessとInsulinへの対応

In [ ]:
sns.histplot(X_train['SkinThickness'], bins=50);

In [ ]:
sns.histplot(X_train['Insulin'], bins=50);

In [ ]:
(X_train['SkinThickness'] == 0).sum()

In [ ]:
(X_train['Insulin'] == 0).sum()

* 欠測値が多すぎるので、同じ一つの値で埋めると、問題あり。

In [ ]:
((X_train['SkinThickness'] == 0) & (X_train['Insulin'] == 0)).sum()

In [ ]:
for i in X_train.index[X_train['SkinThickness'] == 0]:
  if not i in X_train.index[X_train['Insulin'] == 0]:
    print('No')

* SkinThicknessが0の個体は、必ずInsulinも0になっているらしい。

 * ただし、これは訓練データだけでこうなっているだけかもしれないので、この事実に依存して何かをすることはしない。

* 線形回帰でSkinThicknessとInsulinの欠測部分を埋める。
 * 欠測部分を同じ値で埋めたくないため。

* まず、前に使った前処理の関数を別の名前で定義しておく。

In [ ]:
def preprocess_0(X_train, X_valid):
  imp = SimpleImputer(missing_values=0, strategy='median')

  X_train_copy, X_valid_copy = X_train.copy(), X_valid.copy()

  for feature in ['BloodPressure', 'BMI', 'Glucose']:
    X_train_copy[feature] = imp.fit_transform(X_train[[feature]])
    X_valid_copy[feature] = imp.transform(X_valid[[feature]])
    print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

  return X_train_copy, X_valid_copy

* そして、新たに前処理の関数を定義する。
 * この関数の中で、前に使った前処理を定義した関数を呼び出すことにする。

In [ ]:
def preprocess(X_train, X_valid):

  # 前に使った前処理を定義した関数を呼び出す
  X_train_copy, X_valid_copy = preprocess_0(X_train, X_valid)

  # 欠測値を埋めるための回帰において特徴量として使う列
  columns = X_train.columns.drop('SkinThickness').drop('Insulin')

  # 線形回帰で欠測箇所を埋める
  for feature in ['SkinThickness', 'Insulin']:
    reg = LinearRegression()
    indices = (X_train[feature] != 0)
    reg.fit(X_train.loc[indices, columns], X_train.loc[indices, feature])
    prediction = reg.predict(X_train.loc[~indices, columns]).astype(np.int64)
    X_train_copy.loc[~indices, feature] = prediction

    indices = (X_valid[feature] != 0)
    prediction = reg.predict(X_valid.loc[~ indices, columns]).astype(np.int64)
    X_valid_copy.loc[~indices, feature] = prediction

  return X_train_copy, X_valid_copy

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess);

* 次は、k-NNを使って欠測値を埋める。

In [ ]:
# kは、関数の外で値を指定する。

def preprocess(X_train, X_valid):
  # 前に行った前処理を定義した関数を呼び出す
  X_train_copy, X_valid_copy = preprocess_0(X_train, X_valid)

  # 欠測値を埋めるための回帰において特徴量として使う列
  columns = X_train.columns.drop('SkinThickness').drop('Insulin')

  print(f'imputation k-NN k={k}')
  for feature in ['SkinThickness', 'Insulin']:
    reg = KNeighborsRegressor(n_neighbors=k)
    indices = (X_train[feature] != 0)
    reg.fit(X_train.loc[indices, columns], X_train.loc[indices, feature])
    prediction = reg.predict(X_train.loc[~indices, columns]).astype(np.int64)
    X_train_copy.loc[~indices, feature] = prediction

    indices = (X_valid[feature] != 0)
    prediction = reg.predict(X_valid.loc[~indices, columns]).astype(np.int64)
    X_valid_copy.loc[~indices, feature] = prediction

  return X_train_copy, X_valid_copy

In [ ]:
best_k, best_score = 1, 0.0

for k in range(1, 21):
  score = cv(skf, X_train, y_train, preprocess=preprocess)
  print('-'*64)
  if best_score < score:
    best_k, best_score = k, score

print(f'best score {best_score:.4f} for k = {best_k}')

### スケーラー

* k-NNで欠測箇所を埋める関数を別の名前で定義する。

In [ ]:
# best_kは、上で値を設定したグローバル変数。

def preprocess_1(X_train, X_valid):
  # 前に行った前処理を定義した関数を呼び出す
  X_train_copy, X_valid_copy = preprocess_0(X_train, X_valid)

  # 欠測値を埋めるための回帰において特徴量として使う列
  columns = X_train.columns.drop('SkinThickness').drop('Insulin')

  k = best_k
  print(f'imputation k-NN k={k}')
  for feature in ['SkinThickness', 'Insulin']:
    reg = KNeighborsRegressor(n_neighbors=k)
    indices = (X_train[feature] != 0)
    reg.fit(X_train.loc[indices, columns], X_train.loc[indices, feature])
    prediction = reg.predict(X_train.loc[~indices, columns]).astype(np.int64)
    X_train_copy.loc[~indices, feature] = prediction

    indices = (X_valid[feature] != 0)
    prediction = reg.predict(X_valid.loc[~indices, columns]).astype(np.int64)
    X_valid_copy.loc[~indices, feature] = prediction

  return X_train_copy, X_valid_copy

* スケーリングを行う関数の中で、上の関数を呼び出す。

In [ ]:
def preprocess(X_train, X_valid):
  X_train_copy, X_valid_copy = preprocess_1(X_train, X_valid)
  scaler = MinMaxScaler()
  scaler.fit(X_train_copy)
  return scaler.transform(X_train_copy), scaler.transform(X_valid_copy)

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess);

In [ ]:
def preprocess(X_train, X_valid):
  X_train_copy, X_valid_copy = preprocess_1(X_train, X_valid)
  scaler = StandardScaler()
  scaler.fit(X_train_copy)
  return scaler.transform(X_train_copy), scaler.transform(X_valid_copy)

In [ ]:
cv(skf, X_train, y_train, preprocess=preprocess);

### 正則化

In [ ]:
best_C, best_score = 0, 0

for C in np.power(10.0, np.arange(13) - 5):
  score = cv(skf, X_train, y_train, preprocess=preprocess_1, C=C)
  if best_score < score:
    best_C, best_score = C, score
  print('-' * 64)

print(f'best score {best_score:.4f} for C={best_C}')

* 以上をまとめると・・・
 * 'BloodPressure', 'BMI', 'Glucose'の欠損値は中央値で埋める。
 * 'SkinThickness', 'Insulin'の欠損値はk-NNで埋める。
 * ロジスティック回帰はパラメータ`C=0.1`で正則化して使う。

## テストデータで最終評価

* 訓練データの中央値を使って、テストデータの欠測値を埋める。

In [ ]:
# 訓練データについては、最初に取っておいたオリジナル X_train_original を使う。

X_train_copy = X_train_original.copy()
X_test_copy = X_test.copy()

for feature in ['BloodPressure', 'BMI', 'Glucose']:
  imp = SimpleImputer(missing_values=0, strategy='median')
  X_train_copy[feature] = imp.fit_transform(X_train_original[[feature]])
  X_test_copy[feature] = imp.transform(X_test[[feature]])
  print(f'imputation fill value for {feature}: {imp.statistics_[0]}')

* ちゃんと動いたので、上書き。

In [ ]:
X_train = X_train_copy
X_test = X_test_copy

* k-NNでは、上で'BloodPressure', 'BMI', 'Glucose'の欠測値を埋めたデータを使う。

In [ ]:
# 欠測値を埋めるために特徴量として使う列
columns = X_train.columns.drop('SkinThickness').drop('Insulin')

k = best_k

X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

for feature in ['SkinThickness', 'Insulin']:
  reg = KNeighborsRegressor(n_neighbors=k)
  indices = (X_train[feature] != 0)
  reg.fit(X_train.loc[indices, columns], X_train.loc[indices, feature])
  prediction = reg.predict(X_train.loc[~indices, columns]).astype(np.int64)
  X_train_copy.loc[~indices, feature] = prediction
  indices = (X_test[feature] != 0)
  prediction = reg.predict(X_test.loc[~indices, columns]).astype(np.int64)
  X_test_copy.loc[~indices, feature] = prediction

* ちゃんと動いたので上書き。

In [ ]:
X_train = X_train_copy
X_test = X_test_copy

In [ ]:
model = LogisticRegression(max_iter=1000, C=0.1, random_state=123)
model.fit(X_train, y_train)
print('test score: {:.4f}'.format(model.score(X_test, y_test)))

In [ ]:
y_test_pred_proba = model.predict_proba(X_test)
print('ROC AUC: {:.4f}'.format(roc_auc_score(y_test, y_test_pred_proba[:,1])))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(baseline, X_test, y_test, name="baseline", ax=ax)
display = RocCurveDisplay.from_estimator(model, X_test, y_test, name="my model", ax=ax)
_ = display.ax_.set_title("ROC curve")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
PrecisionRecallDisplay.from_estimator(baseline, X_test, y_test, name="baseline", ax=ax)
display = PrecisionRecallDisplay.from_estimator(model, X_test, y_test, name="my model", ax=ax)
_ = display.ax_.set_title("precision-recall curve")

In [ ]:
average_precision = average_precision_score(y_test, y_test_pred_proba[:,1])
print(f'average precision: {average_precision:.4f}')




---



# 課題
* 上の結果を改良できるかどうか、試行錯誤してみてください。
* training setとtest setへの分割は、変更しないでください。
* training set上での試行錯誤は、どんな方法を使ってもいいです。
  * test setは、最終的な性能評価のときに一回使うだけです。